In [12]:
# 齐次矩阵的合成
# 齐次矩阵的的生成可以一个姿态(四元数、欧拉角、轴角、旋转矩阵)和一个平移向量组成
# 先将对应的姿态转成旋转矩阵，然后使用numpy讲旋转矩阵和平移向量填写到齐次矩阵对应的位置

In [11]:
#导入库
import numpy as np
import transforms3d as tfs
# 定义旋转矩阵R和平移向量T
R = np.asarray([[1., 0., 0.],[0., 1., 0.],[0., 0., 1.]])
T = np.asarray([1,0,1])
R,T

(array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]),
 array([1, 0, 1]))

In [12]:
# 旋转矩阵+平移向量

# 使用numpy方法合成齐次变换矩阵
temp = np.hstack((R, T.reshape(3,1))) # 水平拼接：3×3旋转矩阵 + 3×1平移向量 → 3×4矩阵
np.vstack((temp,[0,0,0,1])) # 垂直拼接：3×4矩阵 + [0,0,0,1] → 4×4矩阵

# 使用tfs中的函数合成齐次变换矩阵
tfs.affines.compose(T, R, [1,1,1])

array([[1., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 0., 1., 1.],
       [0., 0., 0., 1.]])

In [13]:
# 四元数+平移向量
# 先将四元数转换成旋转矩阵，然后再合成齐次矩阵

R = tfs.quaternions.quat2mat([1,0,0,0])
tfs.affines.compose(T,R,[1,1,1])

array([[1., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 0., 1., 1.],
       [0., 0., 0., 1.]])

In [14]:
# 齐次矩阵的分解
# 已有齐次矩阵的情况下，将其分解为姿态和平移两部分

In [16]:
# 将qcjz分解为固定轴欧拉角和平移向量
# T[0:3,0:3]：提取旋转矩阵，4×4 齐次矩阵的前 3 行前 3 列是 3×3 的旋转矩阵
# T[:3,3:4]：提取平移向量，4×4 齐次矩阵的前 3 行第 4 列是 3×1 的平移向量
tfs.euler.mat2euler(T[0:3,0:3]),T[:3,3:4]
# 将qcjz分解为四元数和平移向量
tfs.quaternions.mat2quat(T[0:3,0:3]),T[:3,3:4]

(array([1., 0., 0., 0.]),
 array([[1.],
        [0.],
        [1.]]))

In [17]:
# 齐次矩阵的乘法

In [18]:
#1.相机坐标系{C}为参考坐标系，工具坐标系{P}的位置矢量在相机坐标系{C}的x,y,z各轴投影为2,1,2，并且工具坐标系和相机坐标系姿态相同
#2.机器人基坐标系{B}为参考坐标系，相机坐标系{C}的位置矢量在{B}各轴的投影为 0,0,3，坐标系{C}绕着坐标系{B}的x轴转了180度
#求：{B}为参考坐标系，坐标系{P}的位置矢量和旋转矩阵

In [26]:
# T_BP = T_BC ​ T_CP

import math
T_BC = tfs.affines.compose([0,0,3],tfs.euler.euler2mat(math.pi,0,0),[1,1,1])
T_CP = T = tfs.affines.compose([2,1,2],np.identity(3),[1,1,1])
T_BP = np.dot(T_BC,T_CP)
# 分解成欧拉角对比结果
tfs.euler.mat2euler(T_BP[0:3,0:3]),T_BP[:3,3:4]

((3.141592653589793, -0.0, 0.0),
 array([[ 2.],
        [-1.],
        [ 1.]]))

In [27]:
# 齐次矩阵求逆

In [31]:
T_BE = tfs.affines.compose([0.5,0.6,0.8],tfs.quaternions.quat2mat([1,0,0,0]),[1,1,1])
T_CE = tfs.affines.compose([0.00,0.05,0.05],tfs.quaternions.quat2mat([0.707,0.706,0,0]),[1,1,1])
T_CO = tfs.affines.compose([0.00,0.02,0.85],tfs.quaternions.quat2mat([0.877,0.479,0,0]),[1,1,1])
T_EC = np.linalg.inv(T_CE)
np.dot(np.dot(T_BE,T_EC),T_CO)

array([[ 1.        ,  0.        ,  0.        ,  0.5       ],
       [ 0.        ,  0.8421333 ,  0.53926942,  1.39995674],
       [ 0.        , -0.53926942,  0.8421333 ,  0.83113231],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])